In [ ]:
import json
import os
import random
import time

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from modelos_scratch import MultiScaleCNN, TinyYOLOStyle, SharedTrunkMultiTask, CNNWithAttention, PyramidCNN
from dataset_class import MultiTaskObjectDetectionDataset, collate_fn


def compute_attr_acc(preds, targets):
    correct = (preds == targets).sum().item()
    total = len(targets)
    return correct / total if total > 0 else 0


def train_one_model(model, model_name, train_loader, val_loader, device, num_epochs=10):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = []
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        total_loss, total_cls, total_box, total_attr = 0, 0, 0, 0
        batch_times = []
        total_batches = len(train_loader)
        print(f"Epoch {epoch + 1}/{num_epochs} - Total Batches: {total_batches}")

        for batch_idx, (images, targets, attrs) in enumerate(train_loader, start=1):
            start_time = time.time()

            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            attrs = [{k: v.to(device) for k, v in a.items()} for a in attrs]

            optimizer.zero_grad()
            if isinstance(model, SharedTrunkMultiTask):
                loss_dict = model(images, targets, attrs)
                loss = sum(v for v in loss_dict.values())
            else:
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())

            loss.backward()
            optimizer.step()

            batch_time = time.time() - start_time
            batch_times.append(batch_time)
            print(f"Epoch {epoch + 1}, Batch {batch_idx}/{total_batches}: Time = {batch_time:.2f} s")

            total_loss += loss.item()
            total_cls += loss_dict.get("classification_loss", 0).item()
            total_box += loss_dict.get("bbox_regression_loss", 0).item()
            total_attr += loss_dict.get("attribute_loss", 0).item() if "attribute_loss" in loss_dict else 0

        avg_batch_time = sum(batch_times) / len(batch_times) if batch_times else 0

        # Validação simples
        model.eval()
        with torch.no_grad():
            all_w, all_s, all_t = [], [], []
            all_wp, all_sp, all_tp = [], [], []
            for images, targets, attrs in val_loader:
                if not images:
                    continue
                images = [img.to(device) for img in images]
                if isinstance(model, SharedTrunkMultiTask):
                    _, attr_out = model(images, None, None)
                    all_wp += torch.argmax(attr_out["weather"], dim=1).cpu().tolist()
                    all_sp += torch.argmax(attr_out["scene"], dim=1).cpu().tolist()
                    all_tp += torch.argmax(attr_out["timeofday"], dim=1).cpu().tolist()
                    all_w += [a["weather"].item() for a in attrs]
                    all_s += [a["scene"].item() for a in attrs]
                    all_t += [a["timeofday"].item() for a in attrs]

        acc_weather = compute_attr_acc(torch.tensor(all_wp), torch.tensor(all_w)) if all_w else 0
        acc_scene = compute_attr_acc(torch.tensor(all_sp), torch.tensor(all_s)) if all_s else 0
        acc_time = compute_attr_acc(torch.tensor(all_tp), torch.tensor(all_t)) if all_t else 0

        history.append({
            "epoch": epoch + 1,
            "total_batches": total_batches,
            "total_loss": total_loss,
            "classification_loss": total_cls,
            "bbox_loss": total_box,
            "attribute_loss": total_attr,
            "avg_batch_time": avg_batch_time,
            "acc_weather": acc_weather,
            "acc_scene": acc_scene,
            "acc_time": acc_time
        })

        print(
            f"[{model_name}] Epoch {epoch + 1}: loss={total_loss:.4f}, avg batch time={avg_batch_time:.2f} s, attr acc: W={acc_weather:.2%} S={acc_scene:.2%} T={acc_time:.2%}")
        if total_loss < best_val_loss:
            best_val_loss = total_loss
            torch.save(model.state_dict(), f"best_{model_name}.pth")

    # Salvar histórico de treino em arquivo JSON
    os.makedirs("logs", exist_ok=True)
    with open(f"logs/training_log_{model_name}.json", "w") as f:
        json.dump(history, f, indent=2)

    return history


# Caminhos para os dados
splits = {
    "train": ("images/train", "labels/train"),
    "val": ("images/val", "labels/val")
}

# Transforms
transform = transforms.Compose([transforms.ToTensor()])

# Carregamento dos datasets e DataLoaders
datasets, loaders = {}, {}
for split, (img_dir, lbl_dir) in splits.items():
    print(f"📂 Carregando dataset '{split}'...")
    ds = MultiTaskObjectDetectionDataset(img_dir, lbl_dir, transform)
    subset_size = max(1, int(len(ds) * 0.50))
    indices = random.sample(range(len(ds)), subset_size)
    ds_subset = Subset(ds, indices)
    datasets[split] = ds_subset
    loaders[split] = DataLoader(
        ds_subset, batch_size=8, shuffle=(split == "train"),
        collate_fn=collate_fn, num_workers=0
    )

# Mapeamentos
with open("helpers/categories.json", "r") as f:
    category_to_label = json.load(f)
with open("helpers/weather.json", "r") as f:
    weather_to_label = json.load(f)
with open("helpers/scene.json", "r") as f:
    scene_to_label = json.load(f)
with open("helpers/timeofday.json", "r") as f:
    timeofday_to_label = json.load(f)

# Parâmetros
num_classes = len(category_to_label) + 1
num_weather = len(weather_to_label)
num_scene = len(scene_to_label)
num_time = len(timeofday_to_label)

# Disponíveis: multiscale, tinyyolo, sharedtask, attention, pyramid
model_input = input("Modelos disponíveis: multiscale, tinyyolo, sharedtask, attention, pyramid\n "
                    "Digite os modelos a treinar, separados por vírgula (ou 'all' para todos): ").strip().lower()
if model_input == "all" or model_input == "":
    selected_models = ["multiscale", "tinyyolo", "sharedtask", "attention", "pyramid"]
else:
    selected_models = [m.strip() for m in model_input.split(",")]

print(f"Modelos selecionados: {selected_models}")

all_models = {
    "multiscale": MultiScaleCNN(num_classes),
    "tinyyolo": TinyYOLOStyle(num_classes),
    "sharedtask": SharedTrunkMultiTask(num_classes, num_weather, num_scene, num_time),
    "attention": CNNWithAttention(num_classes),
    "pyramid": PyramidCNN(num_classes),
}

models = {k: v for k, v in all_models.items() if k in selected_models}

train_loader, val_loader = loaders["train"], loaders["val"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}
for name, model in models.items():
    print(f"🔧 Treinando modelo: {name}")
    results[name] = train_one_model(model, name, train_loader, val_loader, device, num_epochs=60)

plt.figure(figsize=(12, 6))
for name, hist in results.items():
    plt.plot([h["total_loss"] for h in hist], label=f"{name} - loss")
plt.title("Loss total por modelo")
plt.legend()
plt.grid(True)
plt.savefig("logs/loss_comparison.png")
plt.show()

📂 Carregando dataset 'train'...
Todas as imagens possuem JSON correspondente.
📂 Carregando dataset 'val'...
Todas as imagens possuem JSON correspondente.
Modelos selecionados: ['multiscale', 'tinyyolo', 'sharedtask', 'attention', 'pyramid']
🔧 Treinando modelo: multiscale
Epoch 1/60 - Total Batches: 438
Epoch 1, Batch 1/438: Time = 1.59 s
Epoch 1, Batch 2/438: Time = 0.92 s
Epoch 1, Batch 3/438: Time = 0.98 s
Epoch 1, Batch 4/438: Time = 0.90 s
Epoch 1, Batch 5/438: Time = 1.00 s
Epoch 1, Batch 6/438: Time = 0.95 s
Epoch 1, Batch 7/438: Time = 1.03 s
Epoch 1, Batch 8/438: Time = 1.01 s
